# Análisis y clasificación de textos
**LDS1071 · UDLAP**

Notebook de entrega: utiliza los modelos, las predicciones y los resultados experimentales ya guardados. No descarga el corpus ni vuelve a entrenar. Para reproducir los ajustes desde cero, usar `notebook_autonoma.ipynb`.

## Configuración
Coloca la carpeta `outputs` o `resultados` al lado de esta notebook. En Colab, descomprime el paquete de entrega antes de ejecutar. Carga únicamente modelos Joblib de confianza.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util
from importlib.metadata import version, PackageNotFoundError

COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
INSTALAR = COLAB
DEPENDENCIAS = {"scikit-learn": "1.6.1", "numpy": "2.2.6",
                "scipy": "1.15.2", "joblib": "1.4.2",
                "pandas": "2.2.3", "seaborn": "0.13.2",
                "matplotlib": "3.10.9"}
def instalada(n):
    try:
        return version(n)
    except PackageNotFoundError:
        return None

faltan = [f"{n}=={v}" for n,v in DEPENDENCIAS.items() if instalada(n)!=v]
if faltan and INSTALAR:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*faltan])
    if any(n in sys.modules for n in ("sklearn","numpy","scipy","pandas","joblib","matplotlib","seaborn")):
        raise RuntimeError("Reinicia el kernel después de instalar las dependencias.")
if instalada("scikit-learn")!="1.6.1":
    raise RuntimeError("Los modelos fueron guardados con scikit-learn 1.6.1. Usa el entorno original o activa INSTALAR.")
print("scikit-learn:", instalada("scikit-learn"))

In [ ]:
# Ruta opcional: cambia None si los resultados están en otro directorio.
CARPETA_OUTPUTS = None
base = Path.cwd()
candidatos = ([Path(CARPETA_OUTPUTS).expanduser()] if CARPETA_OUTPUTS else
    [base/"outputs", base/"resultados", base/"outputs"/"resultados", base,
     base/"LDS1071_entrega"/"outputs", base/"LDS1071_entrega"/"resultados"])
R = next((p.resolve() for p in candidatos
          if (p/"experiment001"/"particiones.csv").is_file()), None)
if R is None:
    raise FileNotFoundError("Descomprime la carpeta outputs junto a la notebook o indica CARPETA_OUTPUTS.")
necesarios = [
    "experiment001/particiones.csv", "experiment002/configuraciones.csv",
    "experiment003/predicciones.csv", "experiment003/modelo.joblib",
    "experiment006/predicciones_hibridas.csv", "experiment006/modelo.joblib",
    "experiment007/proyecciones_2d.csv", "experiment008/enlace_SVD-50.csv",
    "experiment009/dbscan.csv", "experiment009/representantes.csv"]
faltantes = [p for p in necesarios if not (R/p).is_file()]
if faltantes:
    raise FileNotFoundError("Faltan resultados guardados: " + ", ".join(faltantes))
print("Resultados de entrada:", R)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, normalized_mutual_info_score)
from scipy.cluster.hierarchy import cut_tree, dendrogram
from matplotlib.lines import Line2D
from time import perf_counter

inicio = perf_counter()
sns.set_theme(style="whitegrid", palette="dark", font="STIXGeneral")
def leer(n,nombre):
    return pd.read_csv(R/f"experiment{n:03d}"/(nombre+".csv"))
def tabla(n,nombre,columnas=None):
    t=leer(n,nombre)
    display((t[columnas] if columnas else t).round(4))
def mostrar(n,*nombres):
    for nombre in nombres:
        display(Image(filename=str(R/f"experiment{n:03d}"/(nombre+".png"))))
%matplotlib inline

## 1. Preparación y representación
Seis categorías, 2,400 documentos de entrenamiento y 900 de prueba. Se recupera el EDA del corpus preprocesado.

In [ ]:
part=leer(1,"particiones")
assert len(part)==3300 and part.huella.is_unique
display(pd.crosstab(part.categoria,part.particion))
tabla(1,"eda")
mostrar(1,"01_categorias","02_longitudes","03_histograma")

### BoW y TF-IDF
Se comparan el vocabulario, la dispersión de las matrices y la varianza retenida.

In [ ]:
tabla(1,"representaciones")
mostrar(1,"04_terminos_BoW","04_terminos_TF-IDF")

In [ ]:
mostrar(1,"05_varianza_svd","06_esparsidad")

## 2. Aprendizaje no supervisado
K-means, aglomerativo, DBSCAN y mezcla gaussiana. La selección original se conserva; para la comparación jerárquica se fija k=10. Las etiquetas solo se utilizan para evaluar.

In [ ]:
config=leer(2,"configuraciones")
mejores=leer(2,"mejores")
ag10=config[(config.algoritmo=="Aglomerativo") & (config.grupos==10)]
comparacion=pd.concat([mejores[mejores.algoritmo!="Aglomerativo"],ag10],ignore_index=True)
display(comparacion[["algoritmo","representacion","parametros","grupos","cobertura","silhouette","NMI"]].round(4))
fig,axs=plt.subplots(1,2,figsize=(12,4))
for ax,y,label in zip(axs,["silhouette","NMI"],["Silueta coseno","NMI"]):
    sns.barplot(data=comparacion,x=y,y="algoritmo",hue="representacion",ax=ax)
    ax.set(xlabel=label,ylabel="")
    ax.legend(title="")
plt.tight_layout();plt.show()

In [ ]:
mostrar(8,"01_kmeans")
mostrar(2,"08_cobertura_dbscan","06_cruzada")

### Reducción dimensional y t-SNE
Las seis proyecciones fueron calculadas a partir de SVD-100. Se cargan las coordenadas guardadas y se conserva la forma de cada clase original.

In [ ]:
tabla(7,"metricas_proyecciones")
coords=leer(7,"proyecciones_2d")
db=leer(9,"dbscan")
L=leer(8,"enlace_SVD-50").to_numpy()
hojas=leer(8,"aglomerativo_SVD-50")
assert hojas.huella.equals(part.huella)
cut=pd.DataFrame({"huella":hojas.huella,"aglomerativo":cut_tree(L,n_clusters=10).ravel()})
assert cut.aglomerativo.nunique()==10
assert len(coords)==19800 and len(db)==3300
clases=sorted(coords.categoria.unique())
pal=sns.color_palette("tab10",10)
colores=dict(zip(clases,pal))
formas=dict(zip(clases,["o","X","^","s","P","D"]))

In [ ]:
def dibujar(ax, d, key, categorias=False):
    for grupo in sorted(d[key].unique()):
        color = colores[grupo] if categorias else ("#b0b0b0" if grupo==-1 else pal[int(grupo)%10])
        for clase in clases:
            q=d[(d[key]==grupo)&(d.categoria==clase)]
            ax.scatter(q.x,q.y,c=[color],marker=formas[clase],s=10,alpha=.25 if grupo==-1 else .65,linewidths=0)
    ax.set(xticks=[],yticks=[])
def leyenda_clases(ax):
    return ax.legend(handles=[Line2D([0],[0],marker=formas[c],color="black",ls="",label=c) for c in clases],
                     loc="upper center",bbox_to_anchor=(.5,-.05),ncol=3,frameon=False)

In [ ]:
fig,axs=plt.subplots(2,3,figsize=(13,7))
for ax,(m,d) in zip(axs.flat,coords.groupby("metodo",sort=False)):
    dibujar(ax,d,"categoria",True); ax.set_title(m)
fig.legend(handles=[Line2D([0],[0],marker=formas[c],color=colores[c],ls="",label=c) for c in clases],loc="lower center",ncol=6,frameon=False)
fig.tight_layout(rect=[0,.06,1,1]);plt.show()

In [ ]:
ts=coords.loc[coords.metodo=="t-SNE"].merge(cut,on="huella",validate="one_to_one").merge(db,on="huella",validate="one_to_one")

In [ ]:
def vista_tsne(key, titulo):
    fig,ax=plt.subplots(figsize=(9,5))
    dibujar(ax,ts,key)
    ax.set(title=titulo,xlim=(ts.x.min()-2,ts.x.max()+2),ylim=(ts.y.min()-2,ts.y.max()+2))
    leg=leyenda_clases(ax); ax.add_artist(leg)
    handles=[Line2D([0],[0],marker="o",color=pal[g],ls="",label=str(g)) for g in range(10)]
    if key=="grupo_dbscan": handles.append(Line2D([0],[0],marker="o",color="#b0b0b0",ls="",label="Ruido"))
    ax.legend(handles=handles,loc="upper center",bbox_to_anchor=(.5,-.17),ncol=6,frameon=False,title="Grupo (módulo 10)" if key=="grupo_dbscan" else "Grupo")
    fig.subplots_adjust(bottom=.25);plt.show()

In [ ]:
vista_tsne("grupo","K-means")

In [ ]:
vista_tsne("aglomerativo","Aglomerativo")

In [ ]:
vista_tsne("grupo_dbscan","DBSCAN")

### Dendrograma coloreado
El color representa la categoría mayoritaria de cada subárbol; la opacidad indica su pureza de Gini normalizada. Las hojas corresponden al corte en diez grupos.

In [ ]:
# Dendrograma: color de clase mayoritaria y opacidad de pureza Gini.
n=len(hojas)
counts=np.zeros((2*n-1,len(clases)),int)
for i,c in enumerate(part.categoria): counts[i,clases.index(c)]=1
for j,(a,b,_,_) in enumerate(L): counts[n+j]=counts[int(a)]+counts[int(b)]
p=counts/counts.sum(axis=1,keepdims=True)
pureza=np.clip(((p**2).sum(axis=1)-1/6)/(1-1/6),0,1)
dd=dendrogram(L,truncate_mode="lastp",p=10,no_plot=True)
pos={node:5+10*i for i,node in enumerate(dd["leaves"])}
height=dict.fromkeys(pos,0.)

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
for j,(a,b,h,_) in enumerate(L):
    a,b=int(a),int(b); node=n+j
    if a not in pos or b not in pos: continue
    x1,x2=pos[a],pos[b]; mid=(x1+x2)/2
    for child,x in [(a,x1),(b,x2)]:
        ax.plot([x,x,mid],[height[child],h,h],color="#cccccc",lw=.7)
        ax.plot([x,x,mid],[height[child],h,h],color=pal[counts[child].argmax()],alpha=pureza[child],lw=2)
    pos[node]=mid;height[node]=h
ax.set(xticks=[5+10*i for i in range(10)],xticklabels=[str(counts[x].sum()) for x in dd["leaves"]],xlabel="Documentos por subárbol",ylabel="Distancia coseno media")
ax.legend(handles=[Line2D([0],[0],color=pal[i],label=c) for i,c in enumerate(clases)],loc="upper center",bbox_to_anchor=(.5,-.17),ncol=3,frameon=False)
plt.show()
print("NMI, corte de 10:",normalized_mutual_info_score(part.categoria,cut.aglomerativo))

### Interpretación de grupos
Los términos de mayor peso TF-IDF y los tres documentos más centrales de cada grupo permiten interpretar su contenido.

In [ ]:
pesos=pd.read_csv(R/"experiment008/pesos_terminos.csv")
fig,axs=plt.subplots(2,5,figsize=(15,6),sharex=True)
for g,ax in enumerate(axs.flat):
    q=pesos.loc[pesos.grupo.eq(g)].nsmallest(5,"rango")
    ax.barh(q.palabra,q.peso_medio_tfidf,color=pal[g]);ax.invert_yaxis();ax.set_title(f"Grupo {g}")
fig.tight_layout();plt.show()

In [ ]:
tabla(9,"representantes",["grupo","rango","categoria","terminos","distancia_coseno_media"])
display(leer(9,"metricas_dbscan").drop(columns=["ARI"],errors="ignore").round(4))

## 3. Aprendizaje supervisado
Siete clasificadores y tres ensambles, seleccionados por F1 macro con cinco pliegues estratificados. Se reutilizan las búsquedas, los tiempos y las predicciones originales.

In [ ]:
v=leer(3,"validacion")
grid=leer(3,"busqueda_cv")
display(grid.groupby("modelo").params.agg(list).to_frame("Rejilla de configuraciones"))
display(v[["modelo","F1_CV","DE_CV","parametros","segundos"]].round(4))

In [ ]:
mostrar(3,"01_validacion","07_estabilidad_cv","08_brecha_cv")

### Evaluación de los diez candidatos
Las métricas se recalculan a partir de las predicciones guardadas de los mismos 900 documentos. No se ajusta ningún estimador.

In [ ]:
def verificar_metricas(n,predicciones,metricas):
    p=leer(n,predicciones)
    m=leer(n,metricas).set_index("modelo")
    prueba=part[part.particion.eq("test")].set_index("huella")
    assert len(p)==900 and p.huella.is_unique and set(p.huella)==set(prueba.index)
    assert np.array_equal(p.real.to_numpy(),prueba.loc[p.huella,"categoria"].to_numpy())
    filas=[]
    for modelo in m.index:
        y,z=p.real,p[modelo]
        filas.append({"modelo":modelo,"Accuracy":accuracy_score(y,z),
            "Precision":precision_score(y,z,average="macro",zero_division=0),
            "Recall":recall_score(y,z,average="macro",zero_division=0),
            "F1":f1_score(y,z,average="macro",zero_division=0)})
    calculadas=pd.DataFrame(filas).set_index("modelo")
    np.testing.assert_allclose(calculadas,m[calculadas.columns],rtol=0,atol=1e-10)
    return p,calculadas.reset_index()
pred,m=verificar_metricas(3,"predicciones","metricas_prueba")
display(v.merge(m,on="modelo")[["modelo","F1_CV","Accuracy","Precision","Recall","F1","segundos"]].round(4))

In [ ]:
fig,axs=plt.subplots(2,5,figsize=(19,8))
for ax,nombre in zip(axs.flat,v.modelo):
    cm=confusion_matrix(pred.real,pred[nombre],labels=clases,normalize="true")
    sns.heatmap(cm,ax=ax,cmap="Blues",vmin=0,vmax=1,annot=True,fmt=".2f",
                cbar=False,xticklabels=clases,yticklabels=clases)
    ax.set_title(nombre);ax.tick_params(axis="x",rotation=90)
plt.tight_layout();plt.show()

In [ ]:
mostrar(3,"04_mejor_confusion","10_f1_por_clase","11_diversidad")

In [ ]:
mostrar(3,"06_tiempos","09_costo_rendimiento")

### Comparación estadística
Bootstrap pareado y estratificado (2,000 réplicas) y ajuste de Holm sobre los tres finalistas por CV.

In [ ]:
tabla(3,"bootstrap_holm")
mostrar(3,"05_bootstrap_holm")

### Extensión con indicadores de grupo
La comparación híbrida reutiliza la misma prueba ya observada, por lo que es exploratoria.

In [ ]:
pred_h,m_h=verificar_metricas(6,"predicciones_hibridas","metricas_hibridas")
tabla(6,"comparacion")
mostrar(6,"01_comparacion_pareada")

In [ ]:
tabla(6,"bootstrap_holm")
mostrar(6,"04_bootstrap_holm")

## 4. Generalización y complejidad
Se modifica una variable a la vez utilizando regresión logística como referencia.

In [ ]:
tabla(4,"representacion",["representacion","mean_train_score","mean_test_score"])
mostrar(4,"01_representacion")

In [ ]:
tabla(4,"dimensionalidad",["caracteristicas_max","mean_test_score"])
mostrar(4,"02_dimensionalidad")

In [ ]:
tabla(8,"componentes_svd",["componentes","mean_test_score"])
mostrar(8,"03_svd")

In [ ]:
tabla(4,"complejidad",["C","error_entrenamiento","error_validacion"])
mostrar(4,"04_complejidad")

## 5. Discusión y clasificación de textos nuevos
Se comparan los candidatos por validación cruzada. Los dos pipelines serializados permiten realizar inferencia sin descargar el corpus ni entrenar.

In [ ]:
print("Modelo original seleccionado por CV:",v.iloc[0].modelo)
display(v.merge(m,on="modelo")[["modelo","F1_CV","F1","segundos"]].round(4))
vh=leer(6,"validacion_hibrida")
print("Mejor modelo híbrido por CV:",vh.iloc[0].modelo)

### Cargar modelos guardados
El modelo híbrido necesita la clase original `GrupoOneHot`. La definición se incluye aquí para deserializarlo sin importar el repositorio.

In [ ]:
import tempfile
# Módulo local que necesita joblib; no ejecuta entrenamiento.
PAQUETE=Path(tempfile.mkdtemp(prefix="lds1071-modelos-"))/"proyecto"
PAQUETE.mkdir()
(PAQUETE/"__init__.py").write_text("",encoding="utf-8")
(PAQUETE/"comun.py").write_text("import re\nSEMILLA = 42\n\ndef limpiar(texto):\n    \"\"\"Quitar URL, correos y espacios; conservar letras y contexto inglés.\"\"\"\n    texto = re.sub(r\"https?://\\S+|www\\.\\S+|\\b\\S+@\\S+\\b\", \" \", texto.lower())\n    return \" \".join(re.findall(r\"[a-z]+(?:'[a-z]+)?\", texto))\n\n",encoding="utf-8")
sys.path.insert(0,str(PAQUETE.parent))

In [ ]:
fuente_temas="\"\"\"Transformación inductiva: etiquetas de grupos ajustadas sin supervisión.\"\"\"\nimport numpy as np\nfrom sklearn.base import BaseEstimator, TransformerMixin, clone\nfrom sklearn.cluster import KMeans\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.pipeline import Pipeline, FeatureUnion\nfrom sklearn.preprocessing import OneHotEncoder, Normalizer, FunctionTransformer\nfrom .comun import SEMILLA\n\nclass GrupoOneHot(TransformerMixin, BaseEstimator):\n    \"\"\"Adaptador mínimo de predict a transform; no utiliza y.\"\"\"\n    def __init__(self, estimador, k=10):\n        self.estimador = estimador\n        self.k = k\n\n    def fit(self, X, y=None):\n        self.estimador_ = clone(self.estimador).fit(X)\n        self.encoder_ = OneHotEncoder(categories=[np.arange(self.k)],\n                                      handle_unknown=\"error\", sparse_output=True)\n        self.encoder_.fit(self.estimador_.predict(X).reshape(-1, 1))\n        return self\n\n    def transform(self, X):\n        return self.encoder_.transform(self.estimador_.predict(X).reshape(-1, 1))\n\ndef caracteristicas(reducir=False, k=10, componentes=50):\n    \"\"\"Conservar exactamente la rama léxica y añadir dos bloques one-hot.\"\"\"\n    lexico = TruncatedSVD(componentes, random_state=SEMILLA) if reducir else FunctionTransformer()\n    temas = Pipeline([\n        (\"svd\", TruncatedSVD(componentes, random_state=SEMILLA)),\n        (\"normalizar\", Normalizer()),\n        (\"grupos\", FeatureUnion([\n            (\"kmeans\", GrupoOneHot(KMeans(k, n_init=10, random_state=SEMILLA), k)),\n            (\"gmm\", GrupoOneHot(GaussianMixture(k, covariance_type=\"diag\", n_init=2,\n                                               random_state=SEMILLA), k)),\n        ])),\n    ])\n    return FeatureUnion([(\"lexico\", lexico), (\"temas\", temas)])\n"
(PAQUETE/"temas.py").write_text(fuente_temas,encoding="utf-8")

In [ ]:
import joblib
from proyecto.comun import limpiar
from proyecto.temas import GrupoOneHot

mejor=joblib.load(R/"experiment003/modelo.joblib")
hibrido=joblib.load(R/"experiment006/modelo.joblib")
assert len(mejor.classes_)==len(hibrido.classes_)==6
print("Los dos pipelines se cargaron correctamente.")

In [ ]:
textos=["The spacecraft entered orbit around Mars.",
        "The hockey team won the championship.",
        "The physician prescribed treatment for the patient."]
limpios=[limpiar(t) for t in textos]
display(pd.DataFrame({"Texto":textos,"Modelo léxico":mejor.predict(limpios),
                      "Modelo híbrido":hibrido.predict(limpios)}))
print(f"Tiempo de análisis desde la carga: {perf_counter()-inicio:.1f} s (sin entrenamiento).")